# Test creating dtopo files in netCDF format

Using the `geoclaw.dtopotools` module from [PR #701](https://github.com/clawpack/geoclaw/pull/701/)

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from clawpack.geoclaw import topotools, dtopotools
import xarray as xr
import rasterio

In [ ]:
dtopo_dir = '/Users/rjl/git/CopesHubTsunamis/dtopo/CSZ_groundmotions/dtopo30sec_nosubevents_kinokada/dtopofiles'
event = 'BL16D'
fname = f'{dtopo_dir}/{event}.dtt3'
dtopo = dtopotools.DTopography(fname, dtopo_type=3)

In [ ]:
fname_nc = f'{event}.nc'
dtopo.write(fname_nc, dtopo_type=4)
print('Created ',fname_nc)

In [ ]:
fname_nc_compressed = f'{event}_compressed.nc'
dtopo.write(fname_nc_compressed, dtopo_type=4, compression=True)
print('Created ',fname_nc_compressed)

In [ ]:
new_dtopo = dtopotools.DTopography(fname_nc_compressed, dtopo_type=4)
print('Reloaded to check that dZ agrees with original')
print(f'dZ has shape {new_dtopo.dZ.shape}')
print(f'at times {new_dtopo.times}')
maxdz = abs(dtopo.dZ - new_dtopo.dZ).max()
print(f'Maximum difference from original is {maxdz:.3e} m')

In [ ]:
print(f'dtopo.x has shape {dtopo.x.shape} from {dtopo.x[0]} to {dtopo.x[-1]}')
print(f'dtopo.y has shape {dtopo.y.shape} from {dtopo.y[0]} to {dtopo.y[-1]}')
print(f'dtopo.times has shape {dtopo.times.shape} from {dtopo.times[0]} to {dtopo.times[-1]}')
print(f'dtopo.dZ has shape {dtopo.dZ.shape}')

## Plot original and version reloaded from .nc

In [ ]:
coast = load('../CSZ_coast.npy')

In [ ]:
# time to plot deformation
#tplot = dtopo.times.max()  # for final static deformation
tplot = 200

fig,axs = subplots(1,2,figsize=(10,6))
for ax in axs:
    ax.plot(coast[:,0], coast[:,1], 'g', linewidth=0.9)
    ax.set_aspect(1/cos(45*pi/180))
    ax.set_xlim(-130,-121)
    ax.set_ylim(39,50)

dtopo.plot_dZ_colors(t=tplot, axes=axs[0], dZ_interval=2, cmax_dZ=10);
axs[0].set_title(f'Seafloor deformation\nat time t = {tplot:.1f} seconds');

new_dtopo.plot_dZ_colors(t=tplot, axes=axs[1], dZ_interval=2, cmax_dZ=10);
axs[1].set_title(f'After reloading\nat time t = {tplot:.1f} seconds');

## Note that significant compression should be possible:

In [ ]:
%%bash
ls -lh BL16D.nc
gzip BL16D.nc
ls -lh BL16D.nc.gz
gunzip BL16D.nc.gz

In [ ]:
fname_nc = f'{event}_compressed.nc'
dtopo.write(fname_nc, dtopo_type=4, compression=True)
print('Created ',fname_nc)

In [ ]:
ls -lh BL16D_compressed.nc

## Read in with rasterio to check metadata

In [ ]:
fname_nc = f'{event}_compressed.nc'
with rasterio.open(fname_nc) as src:
    print(f"Coordinate Reference System (CRS): {src.crs}")
    print(f"Bounds: {src.bounds}")
    print(f"Number of bands: {src.count}")
    print(f"Width/Height: {src.width}x{src.height}")
    
    meta = src.meta
    print('\nsrc.meta = \n')
    for k in meta.keys():
        print(f'{k:<30}{meta[k]}')
    
    tags = src.tags()
    print('\nsrc.tags = \n')
    for k in tags.keys():
        print(f'{k:<30}:  {tags[k]}')

## Check times in xarray

In [ ]:
dtopo_xr = xr.open_dataset(fname_nc)
dtopo_xr

In [ ]:
dtopo_xr.coords['time'][:3]

## Try Audrey's approach to compressing nc file from xarray:

In [ ]:
dtopo_xr.map(lambda a: np.round(a, decimals=3))
comp = dict(zlib=True, complevel=5) #line that specifies compression
encoding = {var: comp for var in dtopo_xr.data_vars}
fname_xr = f'{event}_xrcompressed.nc'
dtopo_xr.to_netcdf(fname_xr)
print('Created ',fname_xr)

In [ ]:
dtopo_xr.close()

In [ ]:
ls -lh BL16D*